# vLLM Tutorial (using Qwen2.5-7B)

Hands-on guide to high-throughput LLM inference with [vLLM](https://docs.vllm.ai/).

**Requirements**: `pip install vllm openai`
**Model**: `Qwen/Qwen2.5-7B-Instruct` (downloaded automatically from HuggingFace)
**Hardware**: NVIDIA GPU with enough VRAM (~16 GB for 7B in float16)

> vLLM loads models directly — no Ollama needed. It uses **PagedAttention** for
> efficient memory management and **continuous batching** for high throughput.

**Covered topics**:
1. Offline Inference
2. Sampling Parameters
3. Chat Completion
4. Batch Inference
5. OpenAI-Compatible API Server
6. Streaming
7. Structured Output (Guided Decoding)

In [ ]:
!pip install -q vllm openai

## 1. Offline Inference

The `LLM` class loads a model into GPU memory and runs inference locally.
This is the simplest way to use vLLM — no server required.

In [ ]:
from vllm import LLM, SamplingParams

MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Load the model (downloaded from HuggingFace on first run)
llm = LLM(
    model=MODEL,
    trust_remote_code=True,
)

# Simple text completion
outputs = llm.generate(["The capital of France is"], SamplingParams(max_tokens=30))

for output in outputs:
    print(f"Prompt:    {output.prompt}")
    print(f"Generated: {output.outputs[0].text}")

## 2. Sampling Parameters

`SamplingParams` controls how tokens are selected during generation.
Key parameters: `temperature`, `top_p`, `top_k`, `max_tokens`,
`repetition_penalty`, `stop`, and `n` (number of sequences).

In [ ]:
prompt = "Write a one-sentence summary of machine learning."

# Deterministic output (greedy decoding)
greedy = SamplingParams(temperature=0, max_tokens=100)
result = llm.generate([prompt], greedy)
print("Greedy (temp=0):")
print(f"  {result[0].outputs[0].text}\n")

# Creative output (high temperature)
creative = SamplingParams(temperature=1.2, top_p=0.95, max_tokens=100)
result = llm.generate([prompt], creative)
print("Creative (temp=1.2):")
print(f"  {result[0].outputs[0].text}\n")

# Nucleus sampling with top_k
nucleus = SamplingParams(temperature=0.7, top_p=0.9, top_k=50, max_tokens=100)
result = llm.generate([prompt], nucleus)
print("Nucleus (top_p=0.9, top_k=50):")
print(f"  {result[0].outputs[0].text}")

In [ ]:
# Generate multiple sequences for the same prompt
multi = SamplingParams(temperature=0.8, max_tokens=60, n=3)
result = llm.generate(["Explain gravity in one sentence."], multi)

print(f"Generated {len(result[0].outputs)} sequences:\n")
for i, out in enumerate(result[0].outputs):
    print(f"  [{i+1}] {out.text.strip()}")

In [ ]:
# Stop sequences and repetition penalty
params = SamplingParams(
    temperature=0,
    max_tokens=200,
    stop=["\n\n", "3."],          # stop at double newline or item 3
    repetition_penalty=1.1,       # penalize repeated tokens
)
result = llm.generate(["List 5 programming languages:\n1."], params)
print(f"Stopped output: 1.{result[0].outputs[0].text}")
print(f"Stop reason: {result[0].outputs[0].stop_reason}")

## 3. Chat Completion

Use `llm.chat()` to send structured messages (system, user, assistant)
instead of raw text. vLLM applies the model's chat template automatically.

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise Python tutor. Answer in 2-3 sentences max."},
    {"role": "user", "content": "What is a decorator?"},
]

params = SamplingParams(temperature=0, max_tokens=150)
result = llm.chat([messages], params)

print(result[0].outputs[0].text)

In [ ]:
# Multi-turn conversation
conversation = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user", "content": "What is the GIL in Python?"},
    {"role": "assistant", "content": "The GIL (Global Interpreter Lock) is a mutex in CPython that allows only one thread to execute Python bytecode at a time, limiting true parallelism for CPU-bound tasks."},
    {"role": "user", "content": "How can I work around it?"},
]

result = llm.chat([conversation], SamplingParams(temperature=0, max_tokens=200))
print(result[0].outputs[0].text)

## 4. Batch Inference

vLLM shines at batch inference — pass multiple prompts and they are processed
together with **continuous batching** for maximum GPU utilization.

In [ ]:
import time

questions = [
    [{"role": "user", "content": "What is a neural network? One sentence."}],
    [{"role": "user", "content": "What is gradient descent? One sentence."}],
    [{"role": "user", "content": "What is backpropagation? One sentence."}],
    [{"role": "user", "content": "What is an activation function? One sentence."}],
    [{"role": "user", "content": "What is a loss function? One sentence."}],
    [{"role": "user", "content": "What is regularization? One sentence."}],
    [{"role": "user", "content": "What is dropout? One sentence."}],
    [{"role": "user", "content": "What is batch normalization? One sentence."}],
]

params = SamplingParams(temperature=0, max_tokens=80)

# Batch inference — all prompts processed together
start = time.perf_counter()
results = llm.chat(questions, params)
elapsed = time.perf_counter() - start

total_tokens = sum(len(r.outputs[0].token_ids) for r in results)
print(f"Batch of {len(questions)} prompts in {elapsed:.2f}s")
print(f"Total output tokens: {total_tokens}")
print(f"Throughput: {total_tokens / elapsed:.1f} tokens/s\n")

for q, r in zip(questions, results):
    print(f"Q: {q[0]['content']}")
    print(f"A: {r.outputs[0].text.strip()}\n")

## 5. OpenAI-Compatible API Server

vLLM can serve models via an **OpenAI-compatible HTTP API**.
Start the server in a terminal, then query it with the `openai` Python client.

```bash
# Start the server (run in a separate terminal)
vllm serve Qwen/Qwen2.5-7B-Instruct --host 0.0.0.0 --port 8000
```

The cells below assume the server is running at `http://localhost:8000`.

In [ ]:
from openai import OpenAI

# Point the OpenAI client at the local vLLM server
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="unused",  # vLLM doesn't require auth by default
)

# List available models
models = client.models.list()
print("Available models:")
for m in models.data:
    print(f"  - {m.id}")

In [ ]:
# Chat completion via the API
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "What is vLLM and why is it fast?"},
    ],
    temperature=0,
    max_tokens=200,
)

print(response.choices[0].message.content)
print(f"\nUsage: {response.usage.prompt_tokens} prompt + {response.usage.completion_tokens} completion tokens")

## 6. Streaming

Streaming returns tokens as they are generated. Works with both the
OpenAI-compatible API (server mode) and offline inference.

In [ ]:
# Streaming via the OpenAI-compatible API
stream = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[
        {"role": "user", "content": "Explain PagedAttention in 3 sentences."},
    ],
    temperature=0,
    max_tokens=200,
    stream=True,
)

print("Streaming response:")
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

## 7. Structured Output (Guided Decoding)

vLLM can constrain generation to match a **JSON schema**, a **regex**,
or a **grammar**. This guarantees valid structured output without retries.

In [ ]:
from vllm import SamplingParams
from vllm.sampling_params import GuidedDecodingParams
import json

# Define the expected JSON schema
movie_schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "year": {"type": "integer"},
        "genre": {"type": "string"},
        "rating": {"type": "number", "minimum": 0, "maximum": 10},
    },
    "required": ["title", "year", "genre", "rating"],
}

params = SamplingParams(
    temperature=0,
    max_tokens=200,
    guided_decoding=GuidedDecodingParams(json=movie_schema),
)

messages = [
    {"role": "user", "content": "Give me info about the movie Inception as JSON."},
]

result = llm.chat([messages], params)
raw = result[0].outputs[0].text
parsed = json.loads(raw)

print(f"Raw output: {raw}")
print(f"Parsed: {parsed}")
print(f"Type: {type(parsed)}")

In [ ]:
# Regex-guided generation: force a date format (YYYY-MM-DD)
params = SamplingParams(
    temperature=0,
    max_tokens=20,
    guided_decoding=GuidedDecodingParams(regex=r"\d{4}-\d{2}-\d{2}"),
)

messages = [
    {"role": "user", "content": "When was Python 3.0 released? Answer with just the date."},
]

result = llm.chat([messages], params)
print(f"Regex-guided output: {result[0].outputs[0].text}")

In [ ]:
# Choice-guided generation: constrain to specific options
params = SamplingParams(
    temperature=0,
    max_tokens=10,
    guided_decoding=GuidedDecodingParams(choice=["positive", "negative", "neutral"]),
)

messages = [
    {"role": "system", "content": "Classify the sentiment of the text."},
    {"role": "user", "content": "I absolutely loved this movie, it was fantastic!"},
]

result = llm.chat([messages], params)
print(f"Sentiment: {result[0].outputs[0].text}")

In [ ]:
# Structured output via the OpenAI-compatible API
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[
        {"role": "user", "content": "Give me info about the movie The Matrix as JSON."},
    ],
    temperature=0,
    max_tokens=200,
    extra_body={
        "guided_json": movie_schema,
    },
)

parsed = json.loads(response.choices[0].message.content)
print(f"API structured output: {parsed}")

## Summary

| Concept | Key API |
|---|---|
| Offline Inference | `LLM`, `llm.generate()` |
| Sampling Parameters | `SamplingParams(temperature, top_p, top_k, n, stop, ...)` |
| Chat Completion | `llm.chat()` (applies chat template) |
| Batch Inference | `llm.generate([prompt1, prompt2, ...])` (continuous batching) |
| API Server | `vllm serve` + `openai.OpenAI(base_url=...)` |
| Streaming | `stream=True` via OpenAI client |
| Structured Output | `GuidedDecodingParams(json=..., regex=..., choice=...)` |

**Key vLLM features**: PagedAttention, continuous batching, tensor parallelism,
guided decoding, OpenAI-compatible API, and support for 100+ model architectures.